# Exhaustive VQE Test: Parameter-Shift with Sampling

Tests parameter-shift gradient with all sampling methods on:
1. **TSP QAOA** (combinatorial optimization)
2. **PUCCD Chemistry** (molecular ground state, IXYZ Hamiltonian)

Gradient methods tested:
- `autograd` (exact, complex128) — ground truth
- `param_shift` + `EFFICIENT_CONTRACTION` (exact, no shots)
- `param_shift` + `RIGHT_SUFFIX_SAMPLING` (sampling, TR topology)
- `param_shift` + `PERFECT_SAMPLING` (sampling, MPS topology)

Topologies: MPS (chain) and TR (ring)

In [ ]:
!pip install -q qiskit qiskit-optimization qiskit-nature pyscf torch numpy pandas matplotlib seaborn hashable_list ordered_set
!pip install -q git+https://github.com/keunjunpark/TREV@real_form_autograd

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', module='scipy.sparse')

import math, time, gc, itertools
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse.linalg as spla

from TREV.circuit import Circuit
from TREV.hamiltonian.hamiltonian import Hamiltonian
from TREV.measure.enums import MeasureMethod
from TREV.transpile import from_qiskit, build_parameter_mapping
from TREV.optimization.gradients.autograd_gradient import AutogradGradient, autograd_gradient
from TREV.optimization.gradients.batch_parameter_shift import BatchParameterShiftGradient, batch_gradient
from TREV.optimization.optimization import minimize as trev_minimize
from TREV.optimization.optimizer import Optimizer
from TREV.measure.right_suffix_sampling import argmax_bitstring_tr_right_suffix

from qiskit.circuit.library import QAOAAnsatz
from qiskit import transpile as qk_transpile
from qiskit.transpiler import CouplingMap

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem_in_bytes / 1e9:.1f} GB')

## Part 1: TSP QAOA

In [ ]:
from qiskit_optimization.applications import Tsp
from qiskit_optimization.converters import QuadraticProgramToQubo

BASIS_GATES_TSP = ['swap', 'rzz', 'rx', 'h']

def make_tsp_hamiltonian(n_cities, seed):
    tsp_inst = Tsp.create_random_instance(n_cities, seed=seed)
    qp = tsp_inst.to_quadratic_program()
    qubo = QuadraticProgramToQubo().convert(qp)
    qubitOp, offset = qubo.to_ising()
    ps, cs = [], []
    for elm in qubitOp:
        ps.append(str(elm.paulis[0][::-1]))
        cs.append(float(elm.coeffs[0].real))
    N = qubitOp.num_qubits
    h = Hamiltonian(N, ps, cs)
    return qubitOp, h, N, tsp_inst

print('TSP helpers loaded.')

In [ ]:
# ── TSP QAOA gradient comparison ──
TSP_CONFIGS = [
    {'nc': 3, 'reps': 1, 'rank': 8, 'seed': 0},
    {'nc': 3, 'reps': 2, 'rank': 8, 'seed': 0},
    {'nc': 3, 'reps': 1, 'rank': 16, 'seed': 0},
]

tsp_rows = []

for cfg in TSP_CONFIGS:
    nc, reps, rank, seed = cfg['nc'], cfg['reps'], cfg['rank'], cfg['seed']
    torch.manual_seed(seed); np.random.seed(seed)

    qubitOp, hamil, N, tsp_inst = make_tsp_hamiltonian(nc, seed)
    qaoa = QAOAAnsatz(qubitOp, reps=reps)
    optimized = qk_transpile(qaoa, optimization_level=3, basis_gates=BASIS_GATES_TSP)

    cm = CouplingMap.from_ring(N)
    routed = qk_transpile(optimized, coupling_map=cm, optimization_level=1,
                          basis_gates=BASIS_GATES_TSP, seed_transpiler=seed)

    circuit, param_base, J, pnames = build_parameter_mapping(
        routed, fuse_zz_swap=True, rank=rank, device=DEVICE)
    K = J.shape[1]
    P = J.shape[0]

    qaoa_x0 = 0.5 * torch.randn(K)
    full_theta = (param_base + J @ qaoa_x0).to(DEVICE)

    label = f'nc={nc} reps={reps} rank={rank}'
    print(f'\n=== TSP {label}: N={N}q, K={K} QAOA -> P={P} TREV ===')

    # Ground truth: autograd (exact, differentiable)
    grad_ad_fn = AutogradGradient(dtype=torch.complex128)
    t0 = time.time()
    g_ref = grad_ad_fn.run(full_theta, circuit, hamil)
    dt_ref = time.time() - t0
    print(f'  {"autograd":>12}: |grad|={g_ref.norm():.4f}, time={dt_ref*1000:.0f}ms')

    # Param-shift methods
    METHODS = [
        ('ps_exact',   MeasureMethod.EFFICIENT_CONTRACTION, 0),
        ('ps_rss_1k',  MeasureMethod.RIGHT_SUFFIX_SAMPLING, 1000),
        ('ps_rss_10k', MeasureMethod.RIGHT_SUFFIX_SAMPLING, 10000),
        ('ps_ps_1k',   MeasureMethod.PERFECT_SAMPLING, 1000),
        ('ps_ps_10k',  MeasureMethod.PERFECT_SAMPLING, 10000),
    ]

    for gname, mm, shots in METHODS:
        grad_fn = BatchParameterShiftGradient(
            shift=math.pi/2, batch_size=None, shots=shots,
            measure_method=mm, depth=1)
        t0 = time.time()
        g = grad_fn.run(full_theta, circuit, hamil)
        dt = time.time() - t0
        cos = torch.nn.functional.cosine_similarity(g_ref.unsqueeze(0), g.unsqueeze(0)).item()
        norm_ratio = g.norm().item() / max(g_ref.norm().item(), 1e-12)
        print(f'  {gname:>12}: |grad|={g.norm():.4f}, cos={cos:.4f}, '
              f'norm_ratio={norm_ratio:.2f}, time={dt*1000:.0f}ms')
        tsp_rows.append({'problem': 'TSP', 'config': label, 'method': gname,
                         'shots': shots, 'cos_vs_autograd': cos,
                         'norm_ratio': norm_ratio, 'time_ms': dt*1000})
        if hasattr(grad_fn, '_gpu_pool') and grad_fn._gpu_pool is not None:
            grad_fn._gpu_pool.shutdown()

    del circuit, full_theta
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    gc.collect()

df_tsp = pd.DataFrame(tsp_rows)
print('\n=== TSP Gradient Summary ===')
display(df_tsp)

## Part 2: PUCCD Chemistry

In [ ]:
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import PUCCD, HartreeFock

jw_mapper = JordanWignerMapper()
BASIS_GATES_CHEM = ['h', 'x', 'cx', 'rz', 'rx']

MOLECULES = {
    'H2':  {'atom': 'H 0 0 0; H 0 0 0.735', 'charge': 0, 'spin': 0},
    'H4':  {'atom': 'H 0 0 0; H 0 0 0.735; H 0 0 1.47; H 0 0 2.205',
            'charge': 0, 'spin': 0},
}

def setup_molecule(name):
    mol = MOLECULES[name]
    driver = PySCFDriver(atom=mol['atom'], charge=mol['charge'],
                         spin=mol['spin'], basis='sto3g')
    problem = driver.run()
    ns = problem.num_spatial_orbitals
    np_ = problem.num_particles
    N = 2 * ns
    nuc_rep = problem.nuclear_repulsion_energy
    qubitOp = jw_mapper.map(problem.hamiltonian.second_q_op())

    mat = qubitOp.to_matrix(sparse=True)
    eigvals, _ = spla.eigsh(mat, k=1, which='SA')
    exact = float(eigvals[0]) + nuc_rep

    ps, cs = [], []
    for elm in qubitOp:
        ps.append(str(elm.paulis[0][::-1]))
        cs.append(float(elm.coeffs[0].real))
    h = Hamiltonian(N, ps, cs)

    print(f'  {name}: {N}q, {len(h.paulis)} terms, has_only_zi={h.has_only_zi}, '
          f'E_exact={exact:.6f} Ha')
    return h, N, ns, np_, nuc_rep, exact


def build_puccd_circuit(name, ns, np_, N, rank):
    hf = HartreeFock(ns, np_, jw_mapper)
    ansatz = PUCCD(ns, np_, jw_mapper, initial_state=hf)
    transpiled = qk_transpile(ansatz, basis_gates=BASIS_GATES_CHEM, optimization_level=0)

    # Route for ring topology
    cm = CouplingMap.from_ring(N)
    routed = qk_transpile(transpiled, coupling_map=cm, optimization_level=1,
                          basis_gates=BASIS_GATES_CHEM, seed_transpiler=42)

    circuit, param_base, jacobian, param_names = build_parameter_mapping(
        routed, fuse_zz_swap=False, rank=rank, device=DEVICE)
    K = jacobian.shape[1]
    P = jacobian.shape[0]
    print(f'    PUCCD rank={rank}: K={K} logical -> P={P} TREV params')
    return circuit, param_base, jacobian, K


mol_data = {}
for name in MOLECULES:
    mol_data[name] = setup_molecule(name)

In [ ]:
# ── PUCCD gradient comparison ──

CHEM_CONFIGS = [
    {'mol': 'H2', 'rank': 4},
    {'mol': 'H2', 'rank': 8},
    {'mol': 'H4', 'rank': 4},
    {'mol': 'H4', 'rank': 8},
]

chem_rows = []

for cfg in CHEM_CONFIGS:
    mol_name, rank = cfg['mol'], cfg['rank']
    h, N, ns, np_, nuc_rep, exact = mol_data[mol_name]

    circuit, param_base, J, K = build_puccd_circuit(mol_name, ns, np_, N, rank)
    P = J.shape[0]

    torch.manual_seed(42)
    x0 = 0.1 * torch.randn(K)
    full_theta = (param_base + J @ x0).to(DEVICE)

    label = f'{mol_name} rank={rank}'
    print(f'\n=== PUCCD {label}: N={N}q, K={K}->{P} params, {len(h.paulis)} H terms ===')

    # Ground truth: autograd
    grad_ad_fn = AutogradGradient(dtype=torch.complex128)
    t0 = time.time()
    g_ref = grad_ad_fn.run(full_theta, circuit, h)
    dt_ref = time.time() - t0
    print(f'  {"autograd":>12}: |grad|={g_ref.norm():.4f}, time={dt_ref*1000:.0f}ms')

    METHODS = [
        ('ps_exact',   MeasureMethod.EFFICIENT_CONTRACTION, 0),
        ('ps_rss_10k', MeasureMethod.RIGHT_SUFFIX_SAMPLING, 10000),
        ('ps_rss_50k', MeasureMethod.RIGHT_SUFFIX_SAMPLING, 50000),
        ('ps_ps_10k',  MeasureMethod.PERFECT_SAMPLING, 10000),
        ('ps_ps_50k',  MeasureMethod.PERFECT_SAMPLING, 50000),
    ]

    for gname, mm, shots in METHODS:
        grad_fn = BatchParameterShiftGradient(
            shift=math.pi/2, batch_size=None, shots=shots,
            measure_method=mm, depth=1)
        t0 = time.time()
        g = grad_fn.run(full_theta, circuit, h)
        dt = time.time() - t0
        cos = torch.nn.functional.cosine_similarity(g_ref.unsqueeze(0), g.unsqueeze(0)).item()
        norm_ratio = g.norm().item() / max(g_ref.norm().item(), 1e-12)
        print(f'  {gname:>12}: |grad|={g.norm():.4f}, cos={cos:.4f}, '
              f'norm_ratio={norm_ratio:.2f}, time={dt*1000:.0f}ms')
        chem_rows.append({'problem': 'PUCCD', 'config': label, 'method': gname,
                          'shots': shots, 'cos_vs_autograd': cos,
                          'norm_ratio': norm_ratio, 'time_ms': dt*1000})
        if hasattr(grad_fn, '_gpu_pool') and grad_fn._gpu_pool is not None:
            grad_fn._gpu_pool.shutdown()

    del circuit, full_theta
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    gc.collect()

df_chem = pd.DataFrame(chem_rows)
print('\n=== PUCCD Gradient Summary ===')
display(df_chem)

## Part 3: VQE Convergence — TSP QAOA

In [ ]:
# ── TSP VQE: autograd vs param_shift sampling methods ──
NC, REPS, RANK, SEED = 3, 1, 8, 0
N_ITERS, LR = 100, 0.05

torch.manual_seed(SEED); np.random.seed(SEED)
qubitOp, hamil, N, tsp_inst = make_tsp_hamiltonian(NC, SEED)
qaoa = QAOAAnsatz(qubitOp, reps=REPS)
optimized = qk_transpile(qaoa, optimization_level=3, basis_gates=BASIS_GATES_TSP)
cm = CouplingMap.from_ring(N)
routed = qk_transpile(optimized, coupling_map=cm, optimization_level=1,
                      basis_gates=BASIS_GATES_TSP, seed_transpiler=SEED)

circuit, param_base, J, pnames = build_parameter_mapping(
    routed, fuse_zz_swap=True, rank=RANK, device=DEVICE)
K = J.shape[1]
qaoa_x0 = 0.5 * torch.randn(K)

VQE_METHODS = [
    ('autograd',   'autograd', None, 0),
    ('ps_exact',   'param_shift', MeasureMethod.EFFICIENT_CONTRACTION, 0),
    ('ps_rss_1k',  'param_shift', MeasureMethod.RIGHT_SUFFIX_SAMPLING, 1000),
    ('ps_ps_1k',   'param_shift', MeasureMethod.PERFECT_SAMPLING, 1000),
]

curves = {}
for label, grad_type, mm, shots in VQE_METHODS:
    print(f'\n--- TSP VQE: {label} ---')
    J_dev = J.to(DEVICE)
    pb_dev = param_base.to(DEVICE)
    qaoa_params = qaoa_x0.clone().to(DEVICE)

    if grad_type == 'autograd':
        grad_fn = AutogradGradient(dtype=torch.complex128)
    else:
        grad_fn = BatchParameterShiftGradient(
            shift=math.pi/2, batch_size=None, shots=shots,
            measure_method=mm, depth=1)

    qaoa_params.requires_grad_(True)
    adam = torch.optim.Adam([qaoa_params], lr=LR)
    evs = []

    with torch.no_grad():
        for ep in range(N_ITERS):
            adam.zero_grad()
            ft = pb_dev + J_dev @ qaoa_params
            fg = grad_fn.run(ft.detach(), circuit, hamil)
            qaoa_params.grad = J_dev.T @ fg
            adam.step()
            ev = circuit.get_expectation_value(ft, hamil, MeasureMethod.EFFICIENT_CONTRACTION)
            evs.append(float(ev.real) if hasattr(ev, 'real') else float(ev))
            if ep % 25 == 0 or ep == N_ITERS-1:
                print(f'  [{ep+1}/{N_ITERS}] loss={evs[-1]:.6f}')

    curves[label] = evs
    if hasattr(grad_fn, '_gpu_pool') and grad_fn._gpu_pool is not None:
        grad_fn._gpu_pool.shutdown()
    del grad_fn
    if DEVICE == 'cuda': torch.cuda.empty_cache()

fig, ax = plt.subplots(figsize=(8, 5))
for label, ev in curves.items():
    ax.plot(ev, label=label, linewidth=2)
ax.set_xlabel('Iteration'); ax.set_ylabel('Expectation Value')
ax.set_title(f'TSP QAOA VQE (nc={NC}, reps={REPS}, rank={RANK})')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Part 4: VQE Convergence — PUCCD Chemistry

In [ ]:
# ── PUCCD VQE: autograd vs param_shift sampling methods ──
CHEM_ACCURACY = 1.6e-3  # Ha
CHEM_ITERS = 200
CHEM_LR = 0.005
CHEM_RANK = 8

for mol_name in ['H2', 'H4']:
    h, N, ns, np_, nuc_rep, exact = mol_data[mol_name]
    circuit, param_base, J_mat, K = build_puccd_circuit(mol_name, ns, np_, N, CHEM_RANK)

    CHEM_VQE_METHODS = [
        ('autograd',   'autograd', None, 0),
        ('ps_exact',   'param_shift', MeasureMethod.EFFICIENT_CONTRACTION, 0),
        ('ps_rss_10k', 'param_shift', MeasureMethod.RIGHT_SUFFIX_SAMPLING, 10000),
        ('ps_ps_10k',  'param_shift', MeasureMethod.PERFECT_SAMPLING, 10000),
    ]

    chem_curves = {}
    for label, grad_type, mm, shots in CHEM_VQE_METHODS:
        print(f'\n--- {mol_name} VQE: {label} ---')
        J_dev = J_mat.to(DEVICE)
        pb_dev = param_base.to(DEVICE)

        torch.manual_seed(42)
        x = 0.01 * torch.randn(K, device=DEVICE)

        if grad_type == 'autograd':
            grad_fn = AutogradGradient(dtype=torch.complex128)
        else:
            grad_fn = BatchParameterShiftGradient(
                shift=math.pi/2, batch_size=None, shots=shots,
                measure_method=mm, depth=1)

        x.requires_grad_(True)
        adam = torch.optim.Adam([x], lr=CHEM_LR)
        evs = []

        with torch.no_grad():
            for ep in range(CHEM_ITERS):
                adam.zero_grad()
                ft = pb_dev + J_dev @ x
                fg = grad_fn.run(ft.detach(), circuit, h)
                x.grad = J_dev.T @ fg
                adam.step()
                ev = circuit.get_expectation_value(ft, h, MeasureMethod.EFFICIENT_CONTRACTION)
                evs.append(float(ev.real) if hasattr(ev, 'real') else float(ev))
                if ep % 50 == 0 or ep == CHEM_ITERS-1:
                    E = evs[-1] + nuc_rep
                    print(f'  [{ep+1}/{CHEM_ITERS}] E={E:.6f} Ha (err={E-exact:+.6f})')

        chem_curves[label] = [e + nuc_rep for e in evs]
        if hasattr(grad_fn, '_gpu_pool') and grad_fn._gpu_pool is not None:
            grad_fn._gpu_pool.shutdown()
        del grad_fn
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    fig, ax = plt.subplots(figsize=(8, 5))
    for label, ev in chem_curves.items():
        ax.plot(ev, label=label, linewidth=2)
    ax.axhline(exact, ls=':', color='black', lw=1.5, label=f'Exact ({exact:.4f})')
    ax.axhline(exact + CHEM_ACCURACY, ls='--', color='gray', lw=1, alpha=0.5, label='Chem. accuracy')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Energy (Ha)')
    ax.set_title(f'{mol_name} PUCCD VQE (N={N}, rank={CHEM_RANK})')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    del circuit
    gc.collect()

## Part 5: Summary

In [ ]:
df_all = pd.concat([df_tsp, df_chem], ignore_index=True)
print('=== Full Gradient Comparison ===')
display(df_all.style.background_gradient(
    subset=['cos_vs_autograd'], cmap='RdYlGn', vmin=0.9, vmax=1.0))